In [1]:
import pandas as pd
import numpy as np
import datetime
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
end_date = pd.to_datetime('2025-12-31 23:59:59')
start_date = pd.to_datetime('2025-01-01')
# end_date = pd.to_datetime(datetime.datetime.today().date())
# start_date = pd.to_datetime(datetime.datetime.today().date() - datetime.timedelta(days=13))
today = datetime.datetime.today().year*10000 + datetime.datetime.today().month*100 + datetime.datetime.today().day

print(f'今日是：{today}')
print(f'start_date：{start_date}')
print(f'end_date：{end_date}')

今日是：20260209
start_date：2025-01-01 00:00:00
end_date：2025-12-31 23:59:59


##### 读取数据，数据清洗(日期格式、'零件号'转换成字符串)，筛选本期数据

In [3]:
mass =  pd.read_excel('./eps3_mass.xlsx')
mass = mass[mass['状态'].isin(['已发布','已确认','已审批'])]
mass['生效日期'] = pd.to_datetime(mass['生效日期'].str[:10])
mass['失效日期'] = pd.to_datetime(mass['失效日期'].str[:10])
mass['创建时间'] = pd.to_datetime(mass['创建时间'].str[:10])
mass['审批通过时间'] = pd.to_datetime(mass['审批通过时间'].str[:10])
mass['零件号'] = mass['零件号'].astype(str)

In [4]:
usefull_mass_column = ['价格通知书号','供应商编码','供应商名称','零件号','零件名称','零件裸价','生效日期','失效日期','地区','创建人','审批通过时间']
df_mass_useful = mass[usefull_mass_column]
df_mass_useful_currently = df_mass_useful[(df_mass_useful['审批通过时间']<=end_date)&(df_mass_useful['审批通过时间']>=start_date)]
df_mass_useful_currently.head(2)
print(f'当期新增零件数{df_mass_useful_currently.shape[0]}')
## 后续如运算可以从此开始

当期新增零件数52924


In [5]:
# 定义价格区间扩展成行(每月)
def expland_monthly(df):
    '价格履历表扩展成每个月'
    date_months = pd.date_range(
        start=df['生效日期'],
        end=df['失效日期'],
        freq='ME'
    )
    return date_months.to_period('M')

##### 同件不同价_颜色件：颜色件零件号最后一位为字母,或最后两位为66；后两位之前一致零件号互为颜色件

In [6]:
# 需考虑如果近期对同一零件号(同一供应商)多次输机

pattern_colour = r'(66)$|([a-zA-Z])$'
df_part_no_colours_currently = df_mass_useful_currently[df_mass_useful_currently['零件号'].str.contains(pattern_colour, regex=True)]
## df_part_no_colours_currently：当期颜色件的价格通知书记录(最后一位为字母、最后两位为66)

df_part_no_colours_currently['基础件号'] = df_part_no_colours_currently['零件号'].str[:-2]

df_part_no_colours_currently_suppliers_colours = (
    df_part_no_colours_currently.drop_duplicates(['供应商编码','基础件号'])
    [['供应商编码','基础件号']]
)
# df_part_no_colours_currently_suppliers_colours：近期输机 供应商编码-基础件号(一一对应的dataframe)


# df_part_no_colours_currently_suppliers_colours.head()
# df_part_no_colours_currently.head(2)

In [7]:
df_price_difference_colours = pd.DataFrame()

for _, combox in df_part_no_colours_currently_suppliers_colours.iterrows():
    
    supplier = combox['供应商编码']
    part_no_colour = combox['基础件号']

    pattern = f'^{re.escape(part_no_colour)}[a-zA-Z0-9]*[a-zA-Z0-9]{{2}}$'

    months_currently = (
    df_part_no_colours_currently[df_part_no_colours_currently['基础件号']==part_no_colour]
    .apply(expland_monthly,axis=1)
    .explode().drop_duplicates()
    )
    # months_currently某个近期输机的基础件号生效的价格月份

    df_mass_useful_colour_in_current = df_mass_useful[(df_mass_useful['零件号'].str.match(pattern,na=False)) & (df_mass_useful['供应商编码']==supplier)]
    # df_mass_useful_colour_in_current:近期输机的某个颜色件(包含其衍生件)全部价格价格履历

    # 当期涉及的颜色件历史价格履历表扩展成每个月
    months = df_mass_useful_colour_in_current.apply(expland_monthly,axis=1)
    df_mass_useful_colour_in_current.loc[:,'month'] = months
    df_mass_useful_colour_in_current_monthly = (
        df_mass_useful_colour_in_current.explode('month')
            .sort_values(by=['零件号','month','审批通过时间'],ascending=False)
            .drop_duplicates(subset=['零件号','供应商编码','month'],keep='first')
        )
    # df_mass_useful_colour_in_current_monthly：近期输机的某个颜色件(包含其衍生件)的全部月度价格记录

    recent_combinations = df_mass_useful_colour_in_current_monthly[['供应商编码', 'month']].drop_duplicates()
    df_price_difference_colour = pd.DataFrame()

    for month in months_currently:
        all_parts_records = df_mass_useful_colour_in_current_monthly[
            (df_mass_useful_colour_in_current_monthly['month'] == month)
        ]

        if len(all_parts_records) > 1:
            unique_price = all_parts_records['零件裸价'].unique()

            if unique_price.size > 1:
                df_price_difference_colour = pd.concat([df_price_difference_colour,all_parts_records],ignore_index=True)


    if len(df_price_difference_colour) > 0:
        df_price_difference_colours = pd.concat([df_price_difference_colours,df_price_difference_colour],ignore_index=True)

if len(df_price_difference_colours) > 0:
    df_price_difference_colours
else:
    print('本期颜色件输机未发现颜色件价格差异')

In [8]:
df_price_difference_colours['基础件号'] = df_price_difference_colours['零件号'].str[:-2]
useful_columns_export = ['价格通知书号','供应商编码','供应商名称','零件号','零件名称','零件裸价','生效日期','失效日期','创建人','基础件号']

In [9]:
df_price_difference_colours_export = (
    df_price_difference_colours[useful_columns_export]
    .drop_duplicates(['价格通知书号','零件号'])
    .sort_values(['基础件号','价格通知书号'],ascending=False)
    )

df_price_difference_colours_export.to_excel(f'./同件不同价清单/本期同件不同价_颜色件_2025_{today}.xlsx',index=False)

## 循环内的测试

In [10]:
# # 上述代码测试
# df_price_difference_colours['零件号'].str[:-2].unique()
# # 循环测试：
# supplier = '400041'
# part_no_colour = 'R1003368'

# months_currently = (
#     df_part_no_colours_currently
#     .apply(expland_monthly,axis=1)
#     .explode().drop_duplicates()
# )

# pattern = f'^{re.escape(part_no_colour)}[a-zA-Z0-9]*[a-zA-Z0-9]{{2}}$'

# df_mass_useful_colour_in_current = df_mass_useful[df_mass_useful['零件号'].str.match(pattern,na=False)]
#     # df_mass_useful_colour_in_current:近期输机的某个颜色件(包含其衍生件)全部价格价格履历

#     # 当期涉及的颜色件历史价格履历表扩展成每个月
# months = df_mass_useful_colour_in_current.apply(expland_monthly,axis=1)
# df_mass_useful_colour_in_current.loc[:,'month'] = months
# df_mass_useful_colour_in_current_monthly = (
#         df_mass_useful_colour_in_current.explode('month')
#             .sort_values(by=['零件号','month','审批通过时间'],ascending=False)
#             .drop_duplicates(subset=['零件号','供应商编码','month'],keep='first')
#         )

# recent_combinations = df_mass_useful_colour_in_current_monthly[['供应商编码', 'month']].drop_duplicates()
# df_price_difference_colour = pd.DataFrame()

# for month in months_currently:
#     all_parts_records = df_mass_useful_colour_in_current_monthly[
#             (df_mass_useful_colour_in_current_monthly['month'] == month)
#         ]
#     if len(all_parts_records) > 1:
#         unique_price = all_parts_records['零件裸价'].unique()
#         if unique_price.size > 1:
#             df_price_difference_colour = pd.concat([df_price_difference_colour,all_parts_records],ignore_index=True)

# if len(df_price_difference_colour)>1:
#     df_price_difference_colour
# else:
#     print('本期颜色件输机未发现颜色件价格差异')

# df_price_difference_colour